# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a set of steps for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(url)

# Show high-level metadata (name and description)
print("Dataset Title: ", dataset.metadata.name)
print("Description: ", dataset.metadata.description)

# Optionally display other metadata fields
print("Published: ", getattr(dataset.metadata, 'datePublished', 'N/A'))
print("License: ", getattr(dataset.metadata, 'license', 'N/A'))
print("Version: ", getattr(dataset.metadata, 'version', 'N/A'))

## 2. Data Overview
Review available record sets, their fields, and their `@id` values as defined by the Croissant schema and accessible through the dataset.

Entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# Discover all available record sets and their fields
print("Available record sets and their fields (by `@id`):\n")
record_sets = dataset.metadata.record_sets
record_set_id_list = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_id_list.append(rs.id)
    print("  Fields (columns):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

# For further exploration, let's pick the first record set's id
main_record_set_id = record_set_id_list[0] if record_set_id_list else None
print(f"Selected RecordSet for demonstration: {main_record_set_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities are referenced by their `@id`.

We'll extract all records for the main record set identified above.

In [ ]:
# Extract data for each selected record set
dataframes = {}
for record_set_id in record_set_id_list:
    print(f"Extracting records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- Columns in {record_set_id}:")
    print(df.columns.tolist())
    print(f"- Number of records: {len(df)}\n")

# Work on the primary/first record set
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Preview of {main_record_set_id}:\n")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter, normalize, group. You must use the field `@id`, as returned in the overview above, to refer to any columns/fields.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id].copy()
    columns = df.columns.tolist()
    print(f"Available columns in {main_record_set_id}:")
    print(columns)

    # Select a numeric field by @id. Example: Find a typical numeric column (e.g., MW, or coverage, etc), fallback if not found.
    sample_numeric_fields = [c for c in columns if any(word in c.lower() for word in ('mw', 'abundance', 'number', 'count', 'coverage', 'peptide', 'intensity', 'score'))]
    numeric_field_id = sample_numeric_fields[0] if sample_numeric_fields else columns[0]
    print(f"Selected numeric field for filtering and normalization: {numeric_field_id}")

    # Remove nulls and filter for meaningful numeric values
    # (try safe conversion in case column is string)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() > 0 else 10
    print(f"Threshold for filtering: {threshold}")
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by another field, choose a categorical field
    categorical_candidates = [c for c in columns if c != numeric_field_id and df[c].dtype == object]
    group_field_id = categorical_candidates[0] if categorical_candidates else None

    if group_field_id:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()  # Mean for demonstration
        print(grouped_df.head(10))
    else:
        print("No categorical field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships. We use the same `@id` references for columns found in the dataset.

Below, a histogram and boxplot for the selected numeric field are shown. A bar plot is provided if a grouping field was available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Defensive select (reuse from EDA)
    sample_numeric_fields = [c for c in df.columns if any(word in c.lower() for word in ('mw', 'abundance', 'number', 'count', 'coverage', 'peptide', 'intensity', 'score'))]
    numeric_field_id = sample_numeric_fields[0] if sample_numeric_fields else df.columns[0]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(4, 5))
    sns.boxplot(y=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.ylabel(numeric_field_id)
    plt.show()

    # Group by categorical field if available
    categorical_candidates = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
    group_field_id = categorical_candidates[0] if categorical_candidates else None
    if group_field_id:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No data for visualization.")

## 6. Conclusion
This notebook demonstrated step-by-step exploration of the FAIR² dataset using the `mlcroissant` library:

- The dataset's structure and metadata were loaded and displayed.
- All core entities (record sets, fields, and columns) were referenced by their Croissant `@id`.
- Data was extracted and loaded into DataFrames for each record set.
- Some basic EDA was performed, including filtering, normalization, and (if possible) grouping using field `@id`s.
- Visualizations were provided for numeric fields, illustrating their distribution and, where possible, group differences.

For additional analysis, consult the dataset and Croissant schema documentation for full field/column definitions and further research directions.